# Final pre-light social/temporal protocol

**Question.** Does either temporal formulation pass the fixed practical criteria, and does a qualifying protocol retain assembly and exact checkpoint behaviour?

**Hypotheses.** Smaller instantaneous-turn steps may stabilize aggregate behaviour; bounded turning may provide a clearer physical response. Neither is assumed superior. Selection is based on temporal agreement and encounter resolution, not prettier schools.

**Controls.** Same social radii, arena, 360° perception, zero noise, synchronous updates, matched seeds, equal 31.25-s comparison horizons and 0.125-s observations. All event-sensitive diagnostics still run every internal update. The fixed criteria were saved before the experiments.

Read notebooks 01–03 first. This notebook recomputes their decisions from raw case summaries. It runs the final 30×180-s cohort **only if** a temporal protocol qualifies. It never adds light.


In [1]:
from pathlib import Path
import sys

MODEL_DIR = next(
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / "papers")
    if (folder / "shiner_model.py").is_file() and (folder / "handlings").is_dir()
)
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
HERE = MODEL_DIR / "handlings" / "trial_three"
STAGE_DIR = HERE

import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(MODEL_DIR))
DATA_DIR = STAGE_DIR / "data"
from handlings.trial_three.temporal_support import compare_adjacent, protocol_candidates
from handlings.trial_two.study_support import run_streamed, load_checkpoint, continuation_digest, state_bytes
criteria = json.loads((DATA_DIR / "predeclared_criteria.json").read_text())
final_criteria = json.loads((DATA_DIR / "final_validation_criteria.json").read_text())
implementation = json.loads((DATA_DIR / "02_implementation_findings.json").read_text())
instant = pd.read_csv(DATA_DIR / "01_case_summary.csv")
bounded = pd.read_csv(DATA_DIR / "03_case_summary.csv")
assert len(instant) == 90 and len(bounded) == 120
assert implementation["AllChecksPassed"]
assert hashlib.sha256((MODEL_DIR / "shiner_model.py").read_bytes()).hexdigest() == criteria["ModelSHA256"]
print("Both fully executed arms and the predeclared criteria are loaded.")


Both fully executed arms and the predeclared criteria are loaded.


## Adjacent-step evidence and selection rule

Primary thresholds apply to matched-cohort ensemble means: absolute changes ≤0.05 in polarization, rotation, largest-group fraction and connected-time fraction; relative changes ≤10% in MeanNND and mean centroid distance. Near-zero distance references use the predeclared absolute fallback. Both candidate and finer mean missed-pass rates must be ≤0.01/model/s, with zero sampled exact overlap. Paired-seed absolute changes and all raw values remain visible as sensitivity diagnostics.

A candidate must pass at all three speeds and all finer available adjacent comparisons. This avoids selecting an accidental agreement at one pair. If both formulations qualify, bounded turning is preferred for its explicit physical angular capacity, not its connected-seed score. Criteria are project conventions, not biological constants, statistical significance tests or a mathematical convergence theorem.


In [2]:
arms = {
    "Instantaneous": (instant, [.03125, .015625, .0078125], None),
    "Bounded 14 rad/s": (bounded, [.0625, .03125, .015625, .0078125], 14.0),
}
comparison_tables = []
metric_tables = []
candidate_rows = []
for name, (cases, dts, rate) in arms.items():
    comparisons, metrics = compare_adjacent(cases, dts)
    candidates = protocol_candidates(comparisons, dts)
    comparison_tables.append(comparisons.assign(Formulation=name))
    metric_tables.append(metrics.assign(Formulation=name))
    candidate_rows.append({"Formulation": name, "AcceptedCandidates": candidates, "MaxTurnRate": rate})
all_comparisons = pd.concat(comparison_tables, ignore_index=True)
all_metrics = pd.concat(metric_tables, ignore_index=True)
all_comparisons.to_csv(DATA_DIR / "04_all_adjacent_decisions.csv", index=False)
all_metrics.to_csv(DATA_DIR / "04_all_metric_decisions.csv", index=False)
display(all_comparisons)
display(pd.DataFrame(candidate_rows))

# Prefer a physically interpretable turn capacity only if it passes the same tests.
selected_protocol = None
for preferred in ("Bounded 14 rad/s", "Instantaneous"):
    candidate = next(row for row in candidate_rows if row["Formulation"] == preferred)
    if candidate["AcceptedCandidates"]:
        selected_protocol = {
            "Formulation": preferred, "dt": candidate["AcceptedCandidates"][0],
            "record_interval": .125, "max_turn_rate": candidate["MaxTurnRate"],
            "fov_degrees": 360.0, "angular_noise": 0.0,
            "repulsion_radius": .5, "orientation_radius": 3.0, "attraction_radius": 5.5,
        }
        break
print("Temporal candidate:", selected_protocol)


,Speed,CandidateDt,FinerDt,Seeds,MetricsPass,FailedMetrics,CandidateMissedPassRate,FinerMissedPassRate,CrossingRateNonIncreasing,CrossingRatePass,ExactOverlapPass,Pass,Formulation
0,1.0,0.031250,0.015625,10,False,"Polarization, ConnectedTimeFraction, MeanNND, ...",0.0000,0.0000,True,True,True,False,Instantaneous
1,1.0,0.015625,0.007812,10,False,"ConnectedTimeFraction, MeanDistanceToCentroid",0.0000,0.0000,True,True,True,False,Instantaneous
2,3.0,0.031250,0.015625,10,False,"MeanNND, MeanDistanceToCentroid",0.0192,0.0096,True,False,True,False,Instantaneous
3,3.0,0.015625,0.007812,10,False,"Polarization, Rotation, ConnectedTimeFraction,...",0.0096,0.0000,True,True,True,False,Instantaneous
4,5.0,0.031250,0.015625,10,False,ConnectedTimeFraction,0.0928,0.0160,True,False,True,False,Instantaneous
5,5.0,0.015625,0.007812,10,False,MeanDistanceToCentroid,0.0160,0.0000,True,False,True,False,Instantaneous
6,1.0,0.062500,0.031250,10,False,"Polarization, Rotation, LargestGroupFraction, ...",0.0000,0.0000,True,True,True,False,Bounded 14 rad/s
7,1.0,0.031250,0.015625,10,False,"Rotation, MeanDistanceToCentroid",0.0000,0.0000,True,True,True,False,Bounded 14 rad/s
8,1.0,0.015625,0.007812,10,False,MeanDistanceToCentroid,0.0000,0.0000,True,True,True,False,Bounded 14 rad/s
9,3.0,0.062500,0.031250,10,False,"ConnectedTimeFraction, MeanDistanceToCentroid",0.0480,0.0128,True,False,True,False,Bounded 14 rad/s


,Formulation,AcceptedCandidates,MaxTurnRate
0,Instantaneous,[],NaN
1,Bounded 14 rad/s,[],14.0


Temporal candidate: None


## Conditional final cohort and regenerated checkpoints

If a temporal candidate qualifies, test 30 random-start seeds for 180 s at speed 1 under that exact protocol. Keep the operational definition of one connected component for at least 5 s. The final-cohort screen, declared before execution, requires at least 90% assembly and at most 10% breakup among successful runs, plus exact checkpoint/source-continuation and paired-control restoration. Report all failures and censoring even if these aggregate screens pass.

Capture the first qualifying state chronologically. Old dt=0.125 checkpoints are not promoted to a newly selected protocol. The legacy defaults remain available and unchanged.


In [3]:
final_rows = []
final_traces = []
checkpoint_rows = []
if selected_protocol is None:
    print("No temporal protocol passed: the conditional 30-seed cohort and new checkpoints are not run.")
else:
    output_dir = DATA_DIR / "final_protocol_checkpoints"
    output_dir.mkdir(exist_ok=True)
    for seed in final_criteria["Seeds"]:
        summary, trace, checkpoint = run_streamed(
            seed=seed, duration=180., speed=1., **{key: value for key, value in selected_protocol.items() if key != "Formulation"}
        )
        final_rows.append(summary)
        final_traces.append(trace.assign(Seed=seed))
        if checkpoint is not None:
            # Validate against the actual next states of the source model, not only two clones.
            loaded = load_checkpoint(checkpoint)
            source = checkpoint["continuation_validation"]
            original_exact = continuation_digest(loaded, source["steps"]) == source["sha256"]
            left, right = load_checkpoint(checkpoint), load_checkpoint(checkpoint)
            paired_exact = True
            for _ in range(64):
                left.step(); right.step()
                paired_exact = paired_exact and state_bytes(left) == state_bytes(right)
            assert original_exact and paired_exact
            (output_dir / f"seed_{seed}.json").write_text(json.dumps(checkpoint, indent=2) + "\n")
            checkpoint_rows.append({"Seed": seed, "OriginalContinuationExact": original_exact,
                                    "PairedControlsExact": paired_exact,
                                    "CheckpointTime": checkpoint["time"], "dt": checkpoint["model_parameters"]["dt"]})
        print("Final-cohort seed", seed, "assembled:", summary["AssemblySuccess"], flush=True)
    pd.DataFrame(final_rows).to_csv(DATA_DIR / "04_final_assembly_cohort.csv", index=False)
    pd.concat(final_traces).to_csv(DATA_DIR / "04_final_assembly_traces.csv", index=False)
    pd.DataFrame(checkpoint_rows).to_csv(DATA_DIR / "04_final_checkpoint_validation.csv", index=False)
    display(pd.DataFrame(final_rows)[["Seed", "AssemblySuccess", "AssemblyTime", "Censored", "PostAssemblyBreakup", "FinalComponentSizes"]])


No temporal protocol passed: the conditional 30-seed cohort and new checkpoints are not run.


In [4]:
temporal_pass = selected_protocol is not None
if final_rows:
    cohort = pd.DataFrame(final_rows)
    successes = int(cohort.AssemblySuccess.sum())
    censored = int(cohort.Censored.sum())
    breakups = int(cohort.loc[cohort.AssemblySuccess, "PostAssemblyBreakup"].sum())
    assembly_pass = successes / len(cohort) >= final_criteria["MinimumAssemblyFraction"]
    stability_pass = successes > 0 and breakups / successes <= final_criteria["MaximumPostAssemblyBreakupFractionAmongAssembled"]
    checkpoint_pass = len(checkpoint_rows) == successes and all(row["OriginalContinuationExact"] and row["PairedControlsExact"] for row in checkpoint_rows)
else:
    successes = censored = breakups = None
    assembly_pass = stability_pass = checkpoint_pass = None
recording_pass = bool(implementation["RecordingStrideValidated"])
ready = temporal_pass and assembly_pass and stability_pass and checkpoint_pass and recording_pass
verdict = "READY FOR LIGHT WITH DOCUMENTED LIMITATIONS" if ready else "NOT READY FOR LIGHT"
report = {
    "Verdict": verdict, "LightImplemented": False,
    "ScientificRuns": len(instant) + len(bounded) + len(final_rows),
    "InstantaneousRuns": len(instant), "BoundedRuns": len(bounded),
    "SelectedProtocol": selected_protocol if ready else None,
    "TemporalCandidate": selected_protocol,
    "FinalValidationRuns": len(final_rows), "Assembled": successes,
    "Censored": censored, "PostAssemblyBreakupRuns": breakups,
    "RegeneratedCheckpoints": len(checkpoint_rows),
    "Gates": {"TemporalStability": temporal_pass, "Assembly": assembly_pass,
              "PostAssemblyStability": stability_pass, "CheckpointRestoration": checkpoint_pass,
              "RecordingIndependent": recording_pass},
    "CandidateTables": candidate_rows,
    "Interpretation": "Practical matched-cohort stability conventions, not formal convergence or proof of biological realism.",
}
(DATA_DIR / "04_final_social_protocol.json").write_text(json.dumps(report, indent=2) + "\n")
display(pd.DataFrame([{"Criterion": key, "Passed": value} for key, value in report["Gates"].items()]))
display(Markdown(f"## Final verdict: **{verdict}**"))
print("Scientific runs:", report["ScientificRuns"], "final validation runs:", len(final_rows), "new checkpoints:", len(checkpoint_rows))


,Criterion,Passed
0,TemporalStability,False
1,Assembly,None
2,PostAssemblyStability,None
3,CheckpointRestoration,None
4,RecordingIndependent,True


## Final verdict: **NOT READY FOR LIGHT**

Scientific runs: 210 final validation runs: 0 new checkpoints: 0


## Answers from the executed evidence

1. **Instantaneous-turn stability:** no all-speed candidate passes. For dt=.015625 → .0078125, mean centroid-distance changes are 37.08%, 52.31% and 27.37% at speeds 1, 3 and 5.
2. **Bounded-turn stability:** some comparisons improve, but it also has no all-speed candidate. In the finest pair, speed 3 passes all gates; speeds 1 and 5 fail. This is not evidence that one formulation is universally superior.
3. **Accepted internal timestep:** none. The finest tested timestep is a comparison reference, not a validated protocol merely because no finer case was run.
4. **Speed 5:** missed-pass resolution improves strongly. At the finest dt both formulations have zero detected missed passes in this sample; however, bounded-turn connected-time fraction changes from .822100 to .659875 and mean group spread changes 23.15%. Numerical acceptability is not established by crossing counts alone.
5. **Final settings:** no new protocol is promoted. The tested bounded convention is 14 rad/s, with dt-dependent cap; both arms retain 360° FOV and zero social noise. A .125-s recording interval is validated separately from the internal dt.
6. **Random-start assembly:** the previous 30/30 result remains evidence for its stored reference protocol. A new-protocol 30-seed cohort was not run because neither temporal arm qualified. No failed-assembly result is inferred from this skip.
7. **Persistent-assembly stability:** likewise untested in the conditional final cohort; previous-stage stability is not transferred automatically to a changed temporal formulation.
8. **Missed interactions:** the finest observed rates are zero in these runs, but the detector excludes reflected paths and zero exact overlaps does not guarantee finite-body separation. These restricted diagnostics cannot replace aggregate temporal-stability evidence.

**Decision: NOT READY FOR LIGHT.** There were 210 full scientific runs (90 instantaneous and 120 bounded), 12 controlled implementation checks, no conditional final-cohort runs and no newly regenerated checkpoints. All previously saved checkpoints remain historical states under their own stored protocol; they are not promoted to official light starting states.

The main remaining measured bottleneck is step-dependent group spread/connectivity despite bounded social turning. Hard zone switching, wall responses, finite-cohort/outlier sensitivity and long-horizon trajectory divergence are possible contributors, not causes established by these experiments. The next investigation should localize these discrepancies and quantify uncertainty rather than tune attraction, noise or acceptance limits for a preferred appearance.


## Scope of the verdict

Answer the eight protocol questions from the displayed data: instantaneous stability; bounded stability; eligible dt versus dt/2; speed-5 encounter resolution; explicit dt/output/turn/FOV/noise settings; new-protocol assembly; post-assembly stability; and missed-pass rarity. A failed temporal gate leaves the cohort and checkpoint gates **untested**, not evidence of failed assembly at that formulation.

Even a passing protocol is limited to this seed sample, horizons, three constant speeds and diagnostic tolerances. Bounded social turning does not cap the separate instantaneous hard-wall reflection. The finest tested step is a numerical comparison, not ground truth. Undirected geometric connectivity does not establish biological schooling. The derived 14-rad/s convention comes from 1.75 rad per 0.125-s reference update; it is not a directly fitted angular-rate parameter.

The task stops at this decision. No light field, scalar light measurement, speed modulation, environmental noise or gradient-performance metric is implemented.
